# Training Model

## Importing libraires and cleaned dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn import metrics
import joblib
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import lightgbm as lgb

In [2]:
df = pd.read_pickle('cleaned_data.pkl')
df.dtypes

Date received                   datetime64[ns]
Product                               category
Sub-product                           category
Issue                                 category
Sub-issue                             category
Consumer complaint narrative            object
Company public response                 object
Company                               category
State                                 category
ZIP code                                object
Consumer consent provided?              object
Submitted via                         category
Date sent to company            datetime64[ns]
Company response to consumer            object
Timely response?                          bool
Consumer disputed?                        bool
Complaint ID                             int64
tag_class                                int64
narrative_length                         int64
is_upheld                                int64
is_financial_relief                      int64
dtype: object

## Model Training

In [3]:
RANDOM_STATE = 42
TEXT_COL = 'Consumer complaint narrative'
TARGET_1 = 'is_upheld'
TARGET_2 = 'is_financial_relief'

In [ ]:
SAMPLE_SIZE = min(60000, len(df))
df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).copy()

# Define categorical columns to include as simple features
CAT_COLS = ['Product', 'Sub-product', 'Issue', 
            'Sub-issue', 'Company', 'State', 'Submitted via']
for c in CAT_COLS:
    if c not in df_sample.columns:
        CAT_COLS = [cc for cc in CAT_COLS if cc in df_sample.columns]
        break
print('Using sample size:', len(df_sample))
print('Categorical columns used:', CAT_COLS)

Using sample size: 60000
Categorical columns used: ['Product', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State', 'Submitted via']


In [ ]:
print('Data shape:', df.shape)
print('Targets distribution:')
print(df[[TARGET_1, TARGET_2]].sum())

Data shape: (903972, 21)
Targets distribution:
is_upheld              856859
is_financial_relief     57089
dtype: int64


## Training - is_upheld

In [ ]:
X = df_sample[[TEXT_COL] + CAT_COLS].copy()
y = df_sample[TARGET_1].astype(int).values
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=RANDOM_STATE, 
                                                    stratify=y)

# convert text to TF-IDF matrix, using unigram and bigrams, keeping terms that 
# exists in atleast 3 documents
text_transformer = TfidfVectorizer(max_features=50000,    # take 50k features
                                   ngram_range=(1,2),     # unigrams and bigrams
                                   min_df=3)              # atleast in 3 documents

# one-hot encoding for categorical columns, it will ignore unseen categories
# during inferencing
cat_transformer = OneHotEncoder(handle_unknown='ignore')

# apply text transformer to text column and categorical transformer to 
# categorical column
preprocessor = ColumnTransformer([
    ('text', text_transformer, TEXT_COL),
    ('cat', cat_transformer, CAT_COLS)
    ], 
    remainder='drop', 
    sparse_threshold=0.3
    )

# Pipeline steps:
# 1. preprocess text and categorical columns
# 2. select minimum 20k features using chi-squared test
# 3. train logistic regression --> 'saga' solver handles large sparse data
pipe = Pipeline([
    ('pre', preprocessor),
    ('select', SelectKBest(chi2, k=min(20000, 200000))),
    ('clf', LogisticRegression(solver='saga', max_iter=1000, 
                               class_weight='balanced',     # handles imbalanced dataset
                               random_state=RANDOM_STATE, 
                               n_jobs=-1))
                               ]
                               )

print(f'Fitting pipeline for target: {TARGET_1}...')
pipe.fit(X_train, y_train)

Fitting pipeline for target: is_upheld...


c:\github_projects\data_science\Classical_ML\financial_complaint_classifier\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,steps,"[('pre', ...), ('select', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# Predict and evaluate
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe, 'predict_proba') else None

res = {}
res['accuracy'] = metrics.accuracy_score(y_test, y_pred)
res['precision'] = metrics.precision_score(y_test, y_pred, zero_division=0)
res['recall'] = metrics.recall_score(y_test, y_pred, zero_division=0)
res['f1'] = metrics.f1_score(y_test, y_pred, zero_division=0)
if y_proba is not None:
    try:
        res['roc_auc'] = metrics.roc_auc_score(y_test, y_proba)
    except Exception:
        res['roc_auc'] = None
else:
    res['roc_auc'] = None

print('Evaluation metrics:')
for k,v in res.items():
    print(f'  {k}: {v}')

Evaluation metrics:
  accuracy: 0.7769166666666667
  precision: 0.9700734658599827
  recall: 0.7890851568679146
  f1: 0.870268960503998
  roc_auc: 0.7543502247762957


In [ ]:
# Save model
model_path = f'{"artifacts/logistic"}_{TARGET_1}.joblib'
joblib.dump(pipe, model_path)
print('Saved model to', model_path)

Saved model to artifacts/logistic_is_upheld.joblib


## Training - is_financial_relief

In [ ]:
df_sample[TARGET_2].value_counts()

is_financial_relief
0    53120
1     3777
Name: count, dtype: int64

In [ ]:
df[df.is_upheld==1].is_financial_relief.value_counts()

is_financial_relief
0    799770
1     57089
Name: count, dtype: int64

In [ ]:
df_sample = df[df.is_upheld==1].copy()
X = df_sample[[TEXT_COL] + CAT_COLS].copy()
y = df_sample[TARGET_2].astype(int).values
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=RANDOM_STATE, 
                                                    stratify=y)

text_transformer = TfidfVectorizer(max_features=5000, 
                                   ngram_range=(1,2), 
                                   min_df=1)
cat_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer([
    ('text', text_transformer, TEXT_COL),
    ('cat', cat_transformer, CAT_COLS)
    ], 
    remainder='drop', 
    sparse_threshold=0.3
    )

# # Pipeline: preprocessing -> feature selection (chi2) -> classifier
# pipe = Pipeline([
#     ('pre', preprocessor),
#     ('select', SelectKBest(chi2, k=min(20000, 200000))),
#     ('clf', LogisticRegression(solver='saga', max_iter=1000, 
#                                class_weight='balanced', 
#                                random_state=RANDOM_STATE, n_jobs=-1))
#                                ]
#                                )
# Pipeline: preprocessing -> feature selection (chi2) -> SMOTE -> classifier
pipe = ImbPipeline([
    ('pre', preprocessor),
    #('select', SelectKBest(chi2, k=1000)),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', lgb.LGBMClassifier(
        n_estimators=500,
        objective='binary',
        metric='auc',
        random_state=RANDOM_STATE
        )
        )
        ]
        )
print(f'Fitting pipeline for target: {TARGET_2}...')
pipe.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe, 'predict_proba') else None

res = {}
res['accuracy'] = metrics.accuracy_score(y_test, y_pred)
res['precision'] = metrics.precision_score(y_test, y_pred, zero_division=0)
res['recall'] = metrics.recall_score(y_test, y_pred, zero_division=0)
res['f1'] = metrics.f1_score(y_test, y_pred, zero_division=0)
if y_proba is not None:
    try:
        res['roc_auc'] = metrics.roc_auc_score(y_test, y_proba)
    except Exception:
        res['roc_auc'] = None
else:
    res['roc_auc'] = None

print('Evaluation metrics:')
for k,v in res.items():
    print(f'  {k}: {v}')

Fitting pipeline for target: is_financial_relief...
[LightGBM] [Info] Number of positive: 639816, number of negative: 639816
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 44.709310 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1163348
[LightGBM] [Info] Number of data points in the train set: 1279632, number of used features: 7061
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\github_projects\data_science\Classical_ML\financial_complaint_classifier\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\github_projects\data_science\Classical_ML\financial_complaint_classifier\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Evaluation metrics:
  accuracy: 0.8937866162500292
  precision: 0.31829869295050356
  recall: 0.5204063758977053
  f1: 0.39500099714152764
  roc_auc: 0.8837619274444013


In [ ]:
# Save model
model_path = f'{"artifacts/logistic"}_{TARGET_2}.joblib'
joblib.dump(pipe, model_path)
print('Saved model to', model_path)

Saved model to artifacts/logistic_is_financial_relief.joblib
